# Augmented GW

**Started:** April 24, 2026

**Last updated:** April 24, 2026

**Research questions:** Does Augmented GW outperform fused GW with orthologs in a shared space?

**Hypothesis:** Yes, because orthologs correlate across species in non one-to-one ways.

**Conclusion:** TBD

**Potential Next Steps:** TBD



### Imports 

In [ ]:
from __future__ import annotations
import numpy as np
from sklearn.decomposition import PCA
import pandas as pd
import scanpy as sc
import anndata as ad
import scipy as sp
from scipy.spatial.distance import cdist
from scipy import sparse
from scipy.stats import pearsonr
import ot
from ot.gromov import entropic_fused_gromov_wasserstein
import matplotlib.pylab as pl
import matplotlib.pyplot as plt
import seaborn as sns
import random
import re
from __future__ import annotations
from typing import Any, Dict, Hashable, Iterable, List, Sequence, Tuple, Union, Optional
from speciesot_helpers import top_n_organisms_from_species, \
                              cell_types_with_n_per_organism, \
                              sample_equal_cell_types, \
                              split_adata_by_celltype_tissue, \
                              match_cells_by_celltype_tissue, \
                              plot_ot_transport_by_celltype, \
                              ortholog_pearson_r2_by_celltype_biomart, \
                              run_entropic_fgw_transport, \
                              ortholog_cell_distances, \
                              mouse_human_orthologs_biomart, \
                              align_adatas_biomart_one2one, \
                              extract_ensembl_expression

import sys
import os

sys.path.append(os.path.abspath("AGW-AISTATS24/src"))

from agw_scootr import agw_scootr

/n/holylabs/mooney_lab/Lab/joshprice/speciesOT/speciesOT_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Load 2000 immmune cells

In [ ]:
MOUSE_H5AD = "/n/holylabs/mooney_lab/Lab/joshprice/speciesOT/data/tabula_muris/sampled_mouse_shared.h5ad"
HUMAN_H5AD = "/n/holylabs/mooney_lab/Lab/joshprice/speciesOT/data/tabula_sapiens/sampled_human_shared.h5ad"

In [ ]:
mouse_full = sc.read_h5ad(MOUSE_H5AD)
human_full = sc.read_h5ad(HUMAN_H5AD)

mouse_B = mouse_full.raw.to_adata()
human_B = human_full.raw.to_adata()

mouse_B.obs = mouse_full.obs.copy()
human_B.obs = human_full.obs.copy()

mouse_B = mouse_B[mouse_B.obs['assay']=="10x 3' v2"]
human_B = human_B[human_B.obs['assay']=="10x 3' v3"]

In [ ]:
mouse_matched_B, human_matched_B = match_cells_by_celltype_tissue(
    mouse_B, human_B,
    cell_type_key="cell_type_ontology_term_id",
    tissue_key="tissue_ontology_term_id",
)

for a in [mouse_matched_B, human_matched_B]:
    a.X = a.X.astype("float32")
    sc.pp.normalize_total(a, target_sum=1e4)
    sc.pp.log1p(a)

In [ ]:
human1 = human_matched_B
mouse1 = mouse_matched_B

In [ ]:
all_orthologs = mouse_human_orthologs_biomart(human1, mouse1)

In [ ]:
correlated_orthologs = ortholog_pearson_r2_by_celltype_biomart(human1, mouse1)

In [ ]:
shared_immune_cell_types = ['T cell','HPSC','macrophage','monocyte','neutrophil','plasma cell','erythrocyte','NK cell','B cell','basophil','dendritic cell']

In [ ]:
mouse1_immune = mouse1[mouse1.obs['shared_cell_type'].isin(shared_immune_cell_types)]
human1_immune = human1[human1.obs['shared_cell_type'].isin(shared_immune_cell_types)]

In [ ]:
idx = np.random.default_rng(42).choice(mouse1_immune.n_obs, 100, replace=False)
mouse1_immune_sub, human1_immune_sub = mouse1_immune[idx].copy(), human1_immune[idx].copy()

In [ ]:
mouse1 = mouse1_immune_sub
human1 = human1_immune_sub

In [ ]:
sc.tl.pca(mouse1)
sc.tl.pca(human1)

In [ ]:
X_mouse_pca = mouse1.obsm['X_pca']
X_human_pca = human1.obsm['X_pca']

In [ ]:
mouse_orths, mask = extract_ensembl_expression(
    mouse1,
    correlated_orthologs['mouse_ensembl_id'],
    gene_key=None,   # or e.g. "gene_ids" if stored there
)

In [ ]:
human_orths, mask = extract_ensembl_expression(
    human1,
    correlated_orthologs['human_ensembl_id'],
    gene_key=None,   # or e.g. "gene_ids" if stored there
)

In [ ]:
def dist(x1, x2=None, metric='euclidean', w=None):
	if x2 is None:
		x2 = x1
	if w is not None:
		return cdist(x1, x2, metric=metric, w=w)
	if metric=="sqeuclidean":
		C=cdist(x1, x2, metric="euclidean")
		return C**2
	return cdist(x1, x2, metric=metric)

In [ ]:
mouse_X = mouse1.X.toarray()
human_X = human1.X.toarray()

In [ ]:
D1 = dist(mouse_X, mouse_X)
D2 = dist(human_X, human_X)

### Run AGW

In [ ]:
Tv, Tc, cost = agw_scootr(mouse_X, human_X, D1, D2, alpha=0.5)

In [ ]:
Tv

In [ ]:
Tc

In [ ]:
# started at 5:21pm